In [1]:
import pandas as pd

df_prod = pd.read_csv("C:/Users/yessine/ai_journey/projects/stage/data/hybrid_manufacturing_categorical.csv")
print(df_prod.columns)

Index(['Job_ID', 'Machine_ID', 'Operation_Type', 'Material_Used',
       'Processing_Time', 'Energy_Consumption', 'Machine_Availability',
       'Scheduled_Start', 'Scheduled_End', 'Actual_Start', 'Actual_End',
       'Job_Status', 'Optimization_Category'],
      dtype='object')


In [2]:
print(df_prod.head())

  Job_ID Machine_ID Operation_Type  Material_Used  Processing_Time  \
0   J001        M01       Grinding           3.17               76   
1   J002        M01       Grinding           3.35               79   
2   J003        M04       Additive           2.29               56   
3   J004        M04       Grinding           1.76              106   
4   J005        M01          Lathe           1.90               46   

   Energy_Consumption  Machine_Availability      Scheduled_Start  \
0               11.42                    96  2023-03-18 08:00:00   
1                6.61                    84  2023-03-18 08:10:00   
2               11.11                    92  2023-03-18 08:20:00   
3               12.50                    95  2023-03-18 08:30:00   
4                8.13                    88  2023-03-18 08:40:00   

         Scheduled_End         Actual_Start           Actual_End Job_Status  \
0  2023-03-18 09:16:00  2023-03-18 08:05:00  2023-03-18 09:21:00  Completed   
1  2023-03-1

In [3]:
print(df_prod.shape)

(1000, 13)


In [4]:
df_prod.isna().sum()


Job_ID                     0
Machine_ID                 0
Operation_Type             0
Material_Used              0
Processing_Time            0
Energy_Consumption         0
Machine_Availability       0
Scheduled_Start            0
Scheduled_End              0
Actual_Start             129
Actual_End               129
Job_Status                 0
Optimization_Category      0
dtype: int64

In [5]:
df_prod.dtypes

Job_ID                    object
Machine_ID                object
Operation_Type            object
Material_Used            float64
Processing_Time            int64
Energy_Consumption       float64
Machine_Availability       int64
Scheduled_Start           object
Scheduled_End             object
Actual_Start              object
Actual_End                object
Job_Status                object
Optimization_Category     object
dtype: object

In [6]:
df_prod["Job_Status"].value_counts()


Job_Status
Completed    673
Delayed      198
Failed       129
Name: count, dtype: int64

In [7]:
df_prod[df_prod["Actual_Start"].isna()]["Job_Status"].value_counts()

Job_Status
Failed    129
Name: count, dtype: int64

In [8]:
df_prod=df_prod[df_prod['Job_Status']=='Completed'].copy()

In [9]:
df_prod['Actual_Start'] = pd.to_datetime(df_prod['Actual_Start'])
df_prod['Actual_End'] = pd.to_datetime(df_prod['Actual_End'])
df_prod['Scheduled_Start'] = pd.to_datetime(df_prod['Scheduled_Start'])
df_prod['Scheduled_End'] = pd.to_datetime(df_prod['Scheduled_End'])

In [10]:
df_prod["Material_Used"].describe()

count    673.000000
mean       3.024279
std        1.143120
min        1.010000
25%        2.040000
50%        3.110000
75%        4.020000
max        5.000000
Name: Material_Used, dtype: float64

In [11]:
df_prod["Processing_Time"].describe()

count    673.00000
mean      71.26003
std       28.39348
min       20.00000
25%       47.00000
50%       73.00000
75%       95.00000
max      120.00000
Name: Processing_Time, dtype: float64

In [13]:
import pandas as pd

# Load
df_prod = pd.read_csv("C:/Users/yessine/ai_journey/projects/stage/data/hybrid_manufacturing_categorical.csv")

# Keep only completed jobs
df_prod = df_prod[df_prod["Job_Status"] == "Completed"].copy()

# Convert datetime columns
datetime_cols = [
    "Scheduled_Start",
    "Scheduled_End",
    "Actual_Start",
    "Actual_End"
]

for col in datetime_cols:
    df_prod[col] = pd.to_datetime(df_prod[col])

# Create production_date (use actual start date)
df_prod["production_date"] = df_prod["Actual_Start"].dt.date

# Create energy intensity KPI
df_prod["energy_intensity"] = (
    df_prod["Energy_Consumption"] / df_prod["Material_Used"]
)

# Rename columns to match ERP schema
df_prod = df_prod.rename(columns={
    "Job_ID": "production_id",
    "Machine_ID": "machine_id",
    "Operation_Type": "operation_type",
    "Material_Used": "quantity_produced",
    "Processing_Time": "processing_time_min",
    "Energy_Consumption": "energy_consumption_kwh",
    "Machine_Availability": "machine_availability_percent",
    "Job_Status": "job_status",
    "Optimization_Category": "optimization_category"
})

# Select final clean columns
production_df = df_prod[[
    "production_id",
    "machine_id",
    "operation_type",
    "quantity_produced",
    "processing_time_min",
    "energy_consumption_kwh",
    "energy_intensity",
    "machine_availability_percent",
    "production_date",
    "optimization_category"
]]

# Save processed production table
production_df.to_csv("data/processed/production.csv", index=False)

In [14]:
production_df.shape

(673, 10)

In [15]:
production_df.head()

,production_id,machine_id,operation_type,quantity_produced,processing_time_min,energy_consumption_kwh,energy_intensity,machine_availability_percent,production_date,optimization_category
0,J001,M01,Grinding,3.17,76,11.42,3.602524,96,2023-03-18,Moderate Efficiency
3,J004,M04,Grinding,1.76,106,12.50,7.102273,95,2023-03-18,Moderate Efficiency
4,J005,M01,Lathe,1.90,46,8.13,4.278947,88,2023-03-18,High Efficiency
5,J006,M02,Additive,4.86,100,13.83,2.845679,86,2023-03-18,Moderate Efficiency
6,J007,M04,Milling,4.67,22,14.20,3.040685,87,2023-03-18,Moderate Efficiency
